# 04 - Revenue Analysis
## NYC Taxi Revenue Optimization Project

**Business Question:**
"Given historical trip data, which zones, hours, and route types
should a NYC taxi fleet prioritize to maximize revenue per driver per shift?"

**Source Table:** `workspace.nyc_taxi.yellow_trips_cleaned` (237.7M trips)

**Analysis Structure:**
1. Overall revenue benchmarks — what does a typical shift earn?
2. Revenue by hour — best and worst hours
3. Revenue by day of week — weekday vs weekend
4. Revenue by zone — top pickup zones
5. Revenue by route type — airport vs city vs outer borough
6. Combined analysis — best zone + hour combinations

**Metric we optimize for:** `revenue_per_hour` — total revenue a driver
can expect per hour they are working, not just per trip.

### Section 1 — Revenue Benchmarks

Before answering which zones and hours are best, we establish
what a typical driver earns as a baseline.

This gives us a reference point for every finding that follows.
A zone is only "good" if it beats this baseline meaningfully.

**Metrics:**
- Avg revenue per trip
- Avg revenue per mile
- Avg revenue per minute
- Avg tip percentage
- Trips per day (proxy for demand)

In [0]:
SELECT
    data_year                               AS year,
    COUNT(*)                                AS total_trips,
    ROUND(AVG(total_amount), 2)             AS avg_rev_per_trip,
    ROUND(AVG(revenue_per_mile), 2)         AS avg_rev_per_mile,
    ROUND(AVG(revenue_per_minute), 2)       AS avg_rev_per_min,
    ROUND(AVG(tip_pct), 2)                  AS avg_tip_pct,
    ROUND(AVG(trip_minutes), 2)             AS avg_trip_mins,
    ROUND(AVG(trip_distance), 2)            AS avg_trip_miles
FROM workspace.nyc_taxi.yellow_trips_cleaned
GROUP BY data_year
ORDER BY data_year;

## Insights Tracker
*Updated continuously — feeds directly into 05_recommendations.py*

---

### INS-001: Major Fare Increase 2022→2023
- Rev/trip jumped 33% ($21.62 → $28.90)
- Rev/mile jumped 37% ($9.35 → $12.85)
- Cause: TLC fare restructure
- Impact: Post-2022 data is not directly comparable to 2019

### INS-002: Revenue Has Stabilized Post-2023
- 2023, 2024, 2025 all within $0.25 of each other
- Fleet can plan around $28.65–$28.90 as reliable per-trip expectation
- Most stable earning environment in the dataset

### INS-003: Tip Percentage Declining
- Peak 20.81% in 2022 → 17.94% in 2025
- Consistent downward trend
- Could indicate app-based tipping fatigue or Uber/Lyft competition

### INS-004: Trips Getting Longer
- Avg trip duration: 14.71 mins (2019) → 17.10 mins (2025)
- Longer trips = higher rev/trip but fewer trips per shift
- Fleet strategy needs to balance trip length vs trip frequency

### INS-005: Trip Volume Not Recovering to 2019 Levels
- 2019: 83.1M trips vs 2025: 39.6M trips
- Market is roughly half pre-COVID size
- Higher per-trip revenue compensating for lower volume

### INS-006: Vendor Reporting Degradation
- 23% null passenger_count in 2025
- Industry-wide issue across both major vendors
- Does not affect revenue analysis — financial columns are clean

### INS-007: Airport Zones Dominate High Revenue
- JFK (132): $71.69 avg total, 9.9M trips
- LaGuardia (138): $57.03 avg total, 6.8M trips
- Newark (1): $100.83 avg total but only 4,651 trips
- Airport trips behave differently from city trips

### INS-008: Evening Hours Have Highest Tip %
- 5pm–7pm: 20.77% tip rate — highest of any window
- 4am–6am: Highest avg total but lower tip %
- Hour of day materially affects driver take-home

### INS-009: Weekday vs Weekend Revenue Gap
- Weekdays (Mon–Thu): 19.3%–19.7% tip rate
- Weekend (Sat–Sun): 18.0%–18.1% tip rate
- Saturday lowest avg total ($22.89) and lowest tip %

### INS-010: 2025 Growing Despite Market Contraction
- 2025 on track for ~48M annualized trips
- Strongest post-COVID year
- Congestion pricing (Jan 2025) may be shifting demand patterns

### Section 2 — Revenue by Hour

We analyze revenue metrics by hour of day across all years.
The goal is to identify which hours generate the most value
per hour worked — not just per trip.

In [0]:
WITH base AS (
    SELECT
        pickup_hour,
        total_amount,
        revenue_per_mile,
        revenue_per_minute,
        tip_pct,
        trip_minutes,
        trip_distance
    FROM workspace.nyc_taxi.yellow_trips_cleaned
    WHERE data_year IN (2023, 2024, 2025)
)

SELECT
    pickup_hour                             AS hour,
    COUNT(*)                                AS total_trips,
    ROUND(AVG(total_amount), 2)             AS avg_rev_per_trip,
    ROUND(AVG(revenue_per_mile), 2)         AS avg_rev_per_mile,
    ROUND(AVG(revenue_per_minute), 2)       AS avg_rev_per_min,
    ROUND(AVG(tip_pct), 2)                  AS avg_tip_pct,
    ROUND(AVG(trip_minutes), 2)             AS avg_trip_mins
FROM base
GROUP BY pickup_hour
ORDER BY pickup_hour;

### Revenue by Hour — Key Findings

- **Best earning window:** 4am–6am ($2.57/min) driven by airport runs
- **Best tip window:** 5pm–7pm (21%+) driven by short city commutes  
- **Avoid:** 12pm–3pm — highest volume but lowest efficiency ($1.76/min)
- **Late night (10pm–12am):** Underrated — solid across all metrics

### Section 3 — Revenue by Zone

We identify which pickup zones generate the most value.
Filtered to 2023–2025 for current relevance.

Minimum 10,000 trips per zone to ensure statistical reliability.

In [0]:
WITH base AS (
    SELECT
        PULocationID,
        total_amount,
        revenue_per_mile,
        revenue_per_minute,
        tip_pct,
        trip_minutes
    FROM workspace.nyc_taxi.yellow_trips_cleaned
    WHERE data_year IN (2023, 2024, 2025)
)

SELECT
    PULocationID                            AS zone_id,
    COUNT(*)                                AS total_trips,
    ROUND(AVG(total_amount), 2)             AS avg_rev_per_trip,
    ROUND(AVG(revenue_per_mile), 2)         AS avg_rev_per_mile,
    ROUND(AVG(revenue_per_minute), 2)       AS avg_rev_per_min,
    ROUND(AVG(tip_pct), 2)                  AS avg_tip_pct,
    ROUND(AVG(trip_minutes), 2)             AS avg_trip_mins
FROM base
GROUP BY PULocationID
HAVING COUNT(*) >= 10000
ORDER BY avg_rev_per_trip DESC
LIMIT 20;

### Revenue by Zone — Key Findings

- **Zone 265** highest avg/trip ($90.05) but only 14K trips — unreliable volume, investigate further
- **JFK (132)** is the most important zone — $82.21/trip, 5.4M trips, reliable high-volume high-revenue
- **LaGuardia (138)** strong at $68.37/trip, 3.6M trips — best tip rate of airport zones (21.12%)
- **Zone 70 (East Harlem)** standout non-airport zone — $65.73/trip, 431K trips, 19.99% tip rate
- **Zone 93** strong at $65.91/trip with good rev/min ($3.16) — worth investigating

**Low tip % warning:**
Zones 86, 117, 51, 55 have near-zero tip rates (under 0.5%) despite decent avg totals.
Likely cash-dominant zones — actual driver revenue may differ from recorded totals.

**INS-011: JFK is the anchor high-revenue zone — 5.4M trips at $82/trip**
No other zone combines volume and revenue at this level.

**INS-012: LaGuardia beats JFK on tip rate — 21% vs 14%**
Shorter trips, better tips. Different driver experience to JFK.

**INS-013: Zone 70 (East Harlem) is the best non-airport zone**
$65.73/trip, 431K trips, 20% tip rate — consistent and high volume.

### Section 4 — Revenue by Route Type

We classify trips into route types based on pickup and dropoff zones.
This tells us which type of trip is most valuable per shift.

**Route Types:**
- Airport pickup → Any dropoff
- City pickup → City dropoff
- City pickup → Airport dropoff
- Outer borough pickup → Any dropoff

In [0]:
WITH base AS (
    SELECT
        CASE
            WHEN PULocationID IN (132, 138, 1)
                THEN 'Airport Pickup'
            WHEN DOLocationID IN (132, 138, 1)
                AND PULocationID NOT IN (132, 138, 1)
                THEN 'City to Airport'
            WHEN PULocationID BETWEEN 1 AND 263
                AND DOLocationID BETWEEN 1 AND 263
                AND PULocationID NOT IN (132, 138, 1)
                AND DOLocationID NOT IN (132, 138, 1)
                THEN 'City to City'
            ELSE 'Other'
        END                         AS route_type,
        total_amount,
        revenue_per_mile,
        revenue_per_minute,
        tip_pct,
        trip_minutes,
        trip_distance
    FROM workspace.nyc_taxi.yellow_trips_cleaned
    WHERE data_year IN (2023, 2024, 2025)
)

SELECT
    route_type,
    COUNT(*)                        AS total_trips,
    ROUND(AVG(total_amount), 2)     AS avg_rev_per_trip,
    ROUND(AVG(revenue_per_mile), 2) AS avg_rev_per_mile,
    ROUND(AVG(revenue_per_minute), 2) AS avg_rev_per_min,
    ROUND(AVG(tip_pct), 2)          AS avg_tip_pct,
    ROUND(AVG(trip_minutes), 2)     AS avg_trip_mins,
    ROUND(AVG(trip_distance), 2)    AS avg_trip_miles
FROM base
GROUP BY route_type
ORDER BY avg_rev_per_trip DESC;

### Revenue by Route Type — Key Findings

- **City to Airport** is the highest value route — $84.72/trip, 18.93% tip rate
- **Airport Pickup** strong at $76.66/trip but lower tip rate (16.89%) than city-to-airport
- **City to City** dominates by volume (103.5M trips) but lowest rev/trip ($23.03)
- **City to City** has best rev/mile ($13.83) and tip rate (19.36%) — high margin, low absolute value

**The core trade-off:**

| Strategy | Route | Rev/Trip | Rev/Min | Trips Available |
|---|---|---|---|---|
| High ticket | City→Airport | $84.72 | $2.52 | 2.5M |
| High volume | City→City | $23.03 | $1.95 | 103.5M |
| Balanced | Airport Pickup | $76.66 | $2.27 | 9.1M |

**INS-014: City to Airport is highest value single route type**
3.7x more revenue per trip than city-to-city. Only 2.5M trips — positioning matters.

**INS-015: City to City dominates volume but not value**
103.5M trips at $23/trip vs 2.5M trips at $84/trip.
A driver doing 3 airport runs earns equivalent to 11 city trips.

**INS-016: Airport Pickup has lowest tip rate of all route types**
Passengers already paid flat-ra

### Section 5 — Combined Zone + Hour Analysis

The most actionable section of the analysis.
We combine zone and hour to find the best zone-hour combinations
a driver should target to maximize revenue per shift.

Filtered to:
- 2023–2025 data only
- Minimum 5,000 trips per combination for reliability
- Top 20 combinations by avg revenue per trip

In [0]:
WITH base AS (
    SELECT
        PULocationID,
        pickup_hour,
        total_amount,
        revenue_per_minute,
        tip_pct,
        trip_minutes,
        CASE
            WHEN PULocationID IN (132, 138, 1) THEN 'Airport'
            ELSE 'City'
        END AS zone_type
    FROM workspace.nyc_taxi.yellow_trips_cleaned
    WHERE data_year IN (2023, 2024, 2025)
)

SELECT
    PULocationID                                AS zone_id,
    zone_type,
    pickup_hour                                 AS hour,
    COUNT(*)                                    AS total_trips,
    ROUND(AVG(total_amount), 2)                 AS avg_rev_per_trip,
    ROUND(AVG(revenue_per_minute), 2)           AS avg_rev_per_min,
    ROUND(AVG(tip_pct), 2)                      AS avg_tip_pct,
    ROUND(AVG(trip_minutes), 2)                 AS avg_trip_mins
FROM base
GROUP BY PULocationID, zone_type, pickup_hour
HAVING COUNT(*) >= 5000
ORDER BY avg_rev_per_trip DESC
LIMIT 20;

### Combined Zone + Hour — Key Findings

- **JFK dominates the top 20 entirely** — no other zone appears in top combinations
- **Best JFK window:** 4pm–7pm ($84–$89/trip) — afternoon peak, highest absolute revenue
- **Most time-efficient JFK window:** Midnight–2am ($2.65–$2.75/min) — shorter trips, faster turnover
- **Least efficient JFK window:** 3pm–4pm ($1.70–$1.75/min) — long trips stuck in traffic

**Two distinct JFK strategies emerge:**

| Strategy | Hours | Rev/Trip | Rev/Min | Trip Length |
|---|---|---|---|---|
| High ticket | 2pm–7pm | $84–$89 | $1.70–$1.99 | 47–57 mins |
| Fast turnover | 10pm–2am | $73–$77 | $2.50–$2.75 | 28–34 mins |

**INS-017: JFK afternoon (2pm–7pm) is highest absolute revenue window**
$84–$89/trip but long trips (47–57 mins) limit daily trip count.

**INS-018: JFK late night (10pm–2am) is most time-efficient airport window**
$2.50–$2.75/min — faster trips mean more trips per shift.

**INS-019: No city zone appears in top 20 zone-hour combinations**
Airport dominates high-revenue combinations entirely.
City zones need separate analysis to find best non-airport opportunities.

### Section 6 — Best Non-Airport Zone + Hour Combinations

JFK dominates the top 20 overall combinations.
Here we isolate city zones to find the best opportunities
for drivers not targeting airport runs.

Filtered to:
- Non-airport pickup zones only
- 2023–2025 data
- Minimum 5,000 trips per combination
- Top 20 by avg revenue per trip

In [0]:
WITH base AS (
    SELECT
        PULocationID,
        pickup_hour,
        total_amount,
        revenue_per_minute,
        tip_pct,
        trip_minutes
    FROM workspace.nyc_taxi.yellow_trips_cleaned
    WHERE data_year IN (2023, 2024, 2025)
      AND PULocationID NOT IN (1, 132, 138)
)

SELECT
    PULocationID                                AS zone_id,
    pickup_hour                                 AS hour,
    COUNT(*)                                    AS total_trips,
    ROUND(AVG(total_amount), 2)                 AS avg_rev_per_trip,
    ROUND(AVG(revenue_per_minute), 2)           AS avg_rev_per_min,
    ROUND(AVG(tip_pct), 2)                      AS avg_tip_pct,
    ROUND(AVG(trip_minutes), 2)                 AS avg_trip_mins
FROM base
GROUP BY PULocationID, pickup_hour
HAVING COUNT(*) >= 5000
ORDER BY avg_rev_per_trip DESC
LIMIT 20;

### Best Non-Airport Zone + Hour — Key Findings

- **Zone 70 (East Harlem) dominates the top 20** — appears 15 out of 20 times
- **Best Zone 70 window:** 4pm–5pm ($68–$70/trip, 20.5% tip rate)
- **Most time-efficient Zone 70:** 9pm–midnight ($2.80–$2.98/min) — short fast trips
- **Zone 93** appears at night only (10pm–1am) — solid at $61–$62/trip, fast turnover
- **Zone 145** single entry at 4pm — $60.93/trip, best rev/min of any zone (3.20/min)

**Two Zone 70 strategies:**

| Strategy | Hours | Rev/Trip | Rev/Min | Trip Length |
|---|---|---|---|---|
| High ticket | 8am–5pm | $67–$70 | $2.05–$2.26 | 32–38 mins |
| Fast turnover | 7pm–midnight | $59–$65 | $2.68–$2.98 | 21–26 mins |

**INS-020: Zone 70 is the JFK equivalent for city-only drivers**
Consistent $60–$70/trip across all hours with 19–21% tip rate.

**INS-021: Zone 70 evening (7pm–midnight) is most time-efficient city window**
$2.68–$2.98/min — comparable to JFK late night efficiency.

**INS-022: Zone 93 is a strong late-night alternative to Zone 70**
$61–$62/trip, $2.72–$2.94/min — worth positioning after 10pm.